# 20 — Lifecycle Feature Extraction

Extract 8 scalar health indicators from every run of one sensor channel, then plot the degradation trajectory over time.

**Dataset**: XJTU-SY Bearing — Bearing 1_1 (35 Hz / 12 kN, 123 run-to-failure snapshots)  
**API**: `assay.lifecycle_features()` · `assay.plot_lifecycle()`

In [ ]:
from pathlib import Path

from isa_phm import ISAWrapper
from bokeh.io import output_notebook
from bokeh.plotting import show as bokeh_show
output_notebook()

# Set this to your dataset's ISA-PHM JSON file.
ISA_JSON = Path("path/to/your-dataset.isa-phm.json").expanduser()
DATA_ROOT = ISA_JSON.parent
print("ISA-JSON:", ISA_JSON)
print("DATA_ROOT:", DATA_ROOT)
print("Exists  :", ISA_JSON.exists())


In [ ]:
wrapper = ISAWrapper(path=ISA_JSON, data_root=DATA_ROOT, strict_validation=False, cache_maxsize=10)

study = wrapper.study("Bearing 1_1")
assay = study.assay(1)          # 1 = horizontal accelerometer (integer index)

print(f"Study : {study.title}")
print(f"Assay : {assay.assay_id}  ({assay.run_count} runs)")


## 1. Extract lifecycle features

`lifecycle_features()` computes 8 scalar statistics per run using a thread-pool.
The result is a tidy DataFrame — one row per run.

| Feature | Sensitivity |
|---------|-------------|
| `rms` | Overall energy — rises steadily with fatigue |
| `kurtosis` | Impulsiveness — spikes sharply near failure |
| `crest_factor` | Peak/RMS — amplifies early micro-pitting |
| `peak2peak` | Dynamic range |
| `std` | Signal variability |
| `skewness` | Asymmetry |
| `mean` | DC offset |
| `max` | Absolute maximum |

In [ ]:
lc = assay.lifecycle_features(file_type="raw", n_workers=8)

print(f"Shape: {lc.shape}")
display(lc.head(8))

## 2. RMS degradation trajectory

RMS is the most reliable early-degradation indicator for rolling-element bearings.

In [ ]:
fig = assay.plot_lifecycle(feature="rms", file_type="raw")
bokeh_show(fig)

## 3. Kurtosis — early fault indicator

Kurtosis rises well before RMS because it measures impulsiveness.  
The sharp early spike (then gradual rise) is the classic bearing fault pattern.

In [ ]:
fig = assay.plot_lifecycle(feature="kurtosis", file_type="raw")
bokeh_show(fig)

## 4. Crest factor — micro-pitting sensitivity

Crest factor (peak / RMS) amplifies very early surface damage.  
Useful as a complementary feature alongside RMS.

In [ ]:
fig = assay.plot_lifecycle(feature="crest_factor", file_type="raw")
bokeh_show(fig)

## 5. Plot all other features

Plot any feature from the lifecycle DataFrame with `plot_lifecycle()`.

In [ ]:
for feature in ["std", "peak2peak", "skewness", "max"]:
    fig = assay.plot_lifecycle(feature=feature, file_type="raw")
    bokeh_show(fig)

## 6. ML-ready lifecycle table

`feature_table()` calculates explicitly chosen features and retains file provenance; researchers must still evaluate modelling suitability.


In [ ]:
ml_df = assay.feature_table(features=["rms", "peak2peak"], file_type="raw", n_workers=8)
print(f"ML table shape: {ml_df.shape}")
print(ml_df.head(5))
